# LeetCode 1981 — Minimize the Difference Between Target and Chosen Elements bằng Python

Link đề: https://leetcode.com/problems/minimize-the-difference-between-target-and-chosen-elements/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def minimizeTheDifference(self, mat: List[List[int]], target: int) -> int:
        ...
```

Bài "chọn mỗi hàng một số" là một dạng **knapsack theo nhóm** trên tập tổng đạt được. Notebook đi từ **thử mọi cách chọn** (vét cạn) → **DP tập tổng** từng hàng → **DP tập tổng có cắt tỉa** (chỉ giữ tổng ≤ target và **một** tổng nhỏ nhất > target) → **DP mảng boolean** → **bitset bằng số nguyên Python** (lời giải chính, rất nhanh).

## 1. Mục tiêu học

1. Mô hình hóa: tập tổng đạt được sau `i` hàng `S_i = {s + x : s ∈ S_{i−1}, x ∈ row_i}`; đáp án `min |s − target|` với `s ∈ S_m`.
2. Thấy giới hạn giá trị nhỏ (`≤ 70`, `m ≤ 70`) ⇒ tổng `≤ 4900` ⇒ DP theo **giá trị tổng** khả thi.
3. Biết **cắt tỉa**: mọi tổng `> target` chỉ cần giữ **cái nhỏ nhất** (các tổng lớn hơn chỉ càng xa target vì giá trị đều dương).
4. Dùng **bitset** (số nguyên Python, phép dịch `<<` và `|`) để cập nhật cả tập tổng một lần cho mỗi phần tử.
5. Kiểm thử bằng vét cạn tích Descartes trên ma trận nhỏ, seed cố định.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `mat` kích thước `m × n` và số nguyên `target`. Chọn **mỗi hàng đúng một số** sao cho **|target − tổng các số đã chọn|** nhỏ nhất. Trả về giá trị nhỏ nhất đó.

**Ví dụ 1:** `mat = [[1,2,3],[4,5,6],[7,8,9]]`, `target = 13` → `0` (chọn `1 + 5 + 7 = 13`).

**Ví dụ 2:** `mat = [[1],[2],[3]]`, `target = 100` → `94` (tổng duy nhất `6`).

**Ví dụ 3:** `mat = [[1,2,9,8,7]]`, `target = 6` → `1` (chọn `7`).

**Ràng buộc:** `1 ≤ m, n ≤ 70`, `1 ≤ mat[i][j] ≤ 70`, `1 ≤ target ≤ 800`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 13, 0), ([[1], [2], [3]], 100, 94), ([[1, 2, 9, 8, 7]], 6, 1)]

for mat, target, exp in examples:
    best = min(product(*mat), key=lambda c: abs(sum(c) - target))
    print(f"target={target}: một cách tốt nhất {best}, tổng {sum(best)}, chênh {abs(sum(best) - target)} (kỳ vọng {exp})")

target=13: một cách tốt nhất (1, 4, 8), tổng 13, chênh 0 (kỳ vọng 0)
target=100: một cách tốt nhất (1, 2, 3), tổng 6, chênh 94 (kỳ vọng 94)
target=6: một cách tốt nhất (7,), tổng 7, chênh 1 (kỳ vọng 1)


## 3. Kiến thức nền

### 3.1 Tích Descartes `itertools.product`

`product(*mat)` sinh mọi bộ (một phần tử mỗi hàng) — `nᵐ` bộ.

In [2]:
print(list(product([1, 2], [3, 4])))
print("số bộ với m = n = 70:", 70 ** 70)

[(1, 3), (1, 4), (2, 3), (2, 4)]
số bộ với m = n = 70: 1435036016098684342856030763566710717400773837392460666392490000000000000000000000000000000000000000000000000000000000000000000000


### 3.2 Tập hợp tổng (subset-sum theo nhóm)

In [3]:
S = {0}
for row in [[1, 2, 3], [4, 5, 6]]:
    S = {s + x for s in S for x in row}
    print("tập tổng:", sorted(S))

tập tổng: [1, 2, 3]
tập tổng: [5, 6, 7, 8, 9]


### 3.3 Bitset bằng số nguyên Python

Bit thứ `s` của số nguyên `B` bằng 1 ⇔ tổng `s` đạt được. "Cộng `x` vào mọi tổng" = `B << x`. Hợp các tập = `|`. Python hỗ trợ số nguyên lớn tùy ý nên một số nguyên làm được bitset hàng nghìn bit, phép dịch chạy bằng C.

In [4]:
B = 1                         # chỉ có tổng 0
row = [1, 2, 3]
nb = 0
for x in row:
    nb |= B << x
B = nb
print(bin(B), "-> các tổng:", [s for s in range(B.bit_length()) if B >> s & 1])

0b1110 -> các tổng: [1, 2, 3]


### 3.4 Tìm bit gần một vị trí

Để tìm tổng đạt được gần `target` nhất: kiểm bit `target`, `target ± 1`, … hoặc liệt kê mọi bit 1 (tổng ≤ 4900 nên nhanh).

In [5]:
B = 0b1011001000
target = 5
sums = [s for s in range(B.bit_length()) if B >> s & 1]
print("tổng đạt được:", sums, "| gần", target, "nhất:", min(sums, key=lambda s: abs(s - target)))

tổng đạt được: [3, 6, 7, 9] | gần 5 nhất: 6


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (`target = 13`):
- Sau hàng 1: `{1, 2, 3}`.
- Sau hàng 2: cộng `{4, 5, 6}` ⇒ `{5, 6, 7, 8, 9}`.
- Sau hàng 3: cộng `{7, 8, 9}` ⇒ `{12, 13, …, 18}`. Có `13` ⇒ đáp án `0`.

Số tổng khác nhau bị chặn bởi `70 · 70 = 4900` — nhỏ hơn rất nhiều so với `nᵐ` cách chọn. Đây là lý do DP theo giá trị tổng hiệu quả.

**Cắt tỉa:** sau mỗi hàng, nếu có nhiều tổng `> target`, chỉ cần giữ **tổng nhỏ nhất** trong số đó — vì về sau tổng chỉ **tăng** (mọi phần tử ≥ 1), một tổng lớn hơn sẽ luôn xa target hơn.

In [6]:
mat, target = [[1, 2, 3], [4, 5, 6], [7, 8, 9]], 13
S = {0}
for i, row in enumerate(mat):
    S = {s + x for s in S for x in row}
    print(f"sau hàng {i}: {sorted(S)}")
print("đáp án:", min(abs(s - target) for s in S))

sau hàng 0: [1, 2, 3]
sau hàng 1: [5, 6, 7, 8, 9]
sau hàng 2: [12, 13, 14, 15, 16, 17, 18]
đáp án: 0


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chọn

In [7]:
def minimize_brute(mat, target):
    return min(abs(sum(c) - target) for c in product(*mat))

for mat_, t_, exp in examples:
    print(minimize_brute(mat_, t_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
94 (kỳ vọng 94 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(nᵐ · m)` — bùng nổ ngay với `m = 10`, `n = 5` (`~10⁷`).

**Nút thắt:** nhiều cách chọn cho **cùng một tổng** — chỉ cần biết tổng nào **đạt được**, không cần biết đạt bằng cách nào.

## 6. Ý tưởng thuật toán / bước đột phá

**DP tập tổng:** `S₀ = {0}`, `S_i = S_{i−1} ⊕ row_i`. Đáp án `min_{s ∈ S_m} |s − target|`. Kích thước `|S_i| ≤ 70i + 1`.

**Cắt tỉa (nếu… thì…):** **nếu** trong `S_i` có các tổng `> target` **thì** chỉ giữ tổng nhỏ nhất trong số đó. Khi đó `|S_i| ≤ target + 1 ≤ 801`.

**Bitset:** biểu diễn `S_i` bằng số nguyên `B`; `B_i = OR_{x ∈ row_i} (B_{i−1} << x)`. Mỗi phần tử một phép dịch + OR trên số tối đa ~4900 bit.

**Bất biến:** sau khi xử lý `i` hàng, tập lưu giữ chứa **mọi** tổng đạt được `≤ target` và (nếu có) tổng đạt được nhỏ nhất `> target`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP tập tổng (không cắt tỉa)

In [8]:
def minimize_set(mat, target):
    S = {0}
    for row in mat:
        S = {s + x for s in S for x in set(row)}
    return min(abs(s - target) for s in S)

for mat_, t_, exp in examples:
    print(minimize_set(mat_, t_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
94 (kỳ vọng 94 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(m · n · |S|)` với `|S| ≤ 4900` ⇒ `~2.4·10⁷` phép toán khi `m = n = 70` — chậm trong Python (vài giây).

### 7.2 Cách 3 — DP tập tổng có cắt tỉa

In [9]:
def minimize_pruned(mat, target):
    S = {0}
    for row in mat:
        row_vals = set(row)
        nxt = set()
        above = None                                  # tổng nhỏ nhất > target
        for s in S:
            for x in row_vals:
                t = s + x
                if t <= target:
                    nxt.add(t)
                elif above is None or t < above:
                    above = t
        if above is not None:
            nxt.add(above)
        S = nxt
    return min(abs(s - target) for s in S)

for mat_, t_, exp in examples:
    print(minimize_pruned(mat_, t_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
94 (kỳ vọng 94 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `|S| ≤ target + 1 ≤ 801` ⇒ `O(m · n · target)` ≈ `4·10⁶`; bộ nhớ `O(target)`.

### 7.3 Cách 4 — DP mảng boolean

In [10]:
def minimize_bool(mat, target):
    f = [True]                                         # f[s] = tổng s đạt được
    for row in mat:
        mx = max(row)
        g = [False] * (len(f) + mx)
        for x in set(row):
            for s in range(len(f)):
                if f[s]:
                    g[s + x] = True
        f = g
    return min(abs(s - target) for s, ok in enumerate(f) if ok)

for mat_, t_, exp in examples:
    print(minimize_bool(mat_, t_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
94 (kỳ vọng 94 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(m · n · 4900)` trong Python thuần — tương tự cách 2. (Trong C++ cách này rất nhanh.)

### 7.4 Cách 5 — Bitset bằng số nguyên Python (lời giải chính)

In [11]:
def minimize_bitset(mat, target):
    B = 1                                              # bit 0: tổng 0
    for row in mat:
        nb = 0
        for x in set(row):
            nb |= B << x
        B = nb
    best = float('inf')
    s = 0
    while B:
        if B & 1:
            best = min(best, abs(s - target))
        B >>= 1; s += 1
    return best

for mat_, t_, exp in examples:
    print(minimize_bitset(mat_, t_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
94 (kỳ vọng 94 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(m · n)` phép dịch/OR trên số `~4900` bit ⇒ thực tế `O(m · n · 4900 / 64)`; bước cuối duyệt `≤ 4900` bit. Nhanh nhất trong Python.

## 8. TRACE từng bước

Trace cách 3 (cắt tỉa) trên `mat = [[1,2,3],[4,5,6],[7,8,9]]` với `target = 10` (ví dụ tự dựng, kiểm bằng vét cạn ở mục 12):

In [12]:
def trace_pruned(mat, target):
    S = {0}
    for i, row in enumerate(mat):
        allsums = sorted({s + x for s in S for x in row})
        keep = [t for t in allsums if t <= target]
        above = [t for t in allsums if t > target]
        S = set(keep + above[:1])
        print(f"  hàng {i} {row}: mọi tổng {allsums} -> giữ {sorted(S)} (bỏ {above[1:]})")
    ans = min(abs(s - target) for s in S)
    print("  đáp án:", ans)
    return ans

trace_pruned([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 10)

  hàng 0 [1, 2, 3]: mọi tổng [1, 2, 3] -> giữ [1, 2, 3] (bỏ [])
  hàng 1 [4, 5, 6]: mọi tổng [5, 6, 7, 8, 9] -> giữ [5, 6, 7, 8, 9] (bỏ [])
  hàng 2 [7, 8, 9]: mọi tổng [12, 13, 14, 15, 16, 17, 18] -> giữ [12] (bỏ [13, 14, 15, 16, 17, 18])
  đáp án: 2


2

In [13]:
def trace_bitset(mat, target):
    B = 1
    for i, row in enumerate(mat):
        nb = 0
        for x in set(row):
            nb |= B << x
        B = nb
        print(f"  hàng {i}: B = {bin(B)} -> tổng {[s for s in range(B.bit_length()) if B >> s & 1]}")
    return B

_ = trace_bitset([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 13)

  hàng 0: B = 0b1110 -> tổng [1, 2, 3]
  hàng 1: B = 0b1111100000 -> tổng [5, 6, 7, 8, 9]
  hàng 2: B = 0b1111111000000000000 -> tổng [12, 13, 14, 15, 16, 17, 18]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minimizeTheDifference(self, mat: List[List[int]], target: int) -> int:
        B = 1                                  # bitset các tổng đạt được
        for row in mat:
            nb = 0
            for x in set(row):
                nb |= B << x
            B = nb
        best, s = float('inf'), 0
        while B:
            if B & 1:
                d = abs(s - target)
                if d < best:
                    best = d
            B >>= 1
            s += 1
        return best

for mat_, t_, exp in examples:
    assert Solution().minimizeTheDifference(mat_, t_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**DP tập tổng.** Quy nạp: `S_i` là đúng tập các tổng của mọi cách chọn trên `i` hàng đầu: một cách chọn trên `i` hàng = cách chọn trên `i − 1` hàng (tổng `s ∈ S_{i−1}`) cộng một phần tử `x` của hàng `i`. Bitset `B_i` mã hóa đúng `S_i` vì `B << x` là tập `{s + x}` và `|` là phép hợp.

**Cắt tỉa.** Gọi `a` là tổng nhỏ nhất `> target` trong `S_i` và `b > a` là một tổng khác `> target`. Mọi phần mở rộng của `b` (cộng thêm các phần tử ≥ 1) cho tổng `b + r > a + r > target` ⇒ `|b + r − target| > |a + r − target|`. Vậy bỏ `b` không làm mất đáp án tối ưu. Các tổng `≤ target` đều phải giữ (về sau có thể tiến tới gần target).

**Đáp án** là `min |s − target|` trên tập cuối — đúng định nghĩa.

## 11. Phân tích Time / Space Complexity

Gọi `m, n` là kích thước, `V = 70` (giá trị lớn nhất), `T = target`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(nᵐ · m)` | `O(m)` |
| 2. Tập tổng | `O(m · n · mV)` | `O(mV)` |
| 3. Tập tổng cắt tỉa | `O(m · n · T)` | `O(T)` |
| 4. Mảng boolean | `O(m · n · mV)` | `O(mV)` |
| 5. Bitset | `O(m · n · mV / 64)` | `O(mV)` bit |

In [15]:
rnd = random.Random(1981)
big = [[rnd.randint(1, 70) for _ in range(70)] for _ in range(70)]
mid = [row[:30] for row in big[:30]]
for name, fn, data in [("pruned 70x70", minimize_pruned, big), ("bitset 70x70", minimize_bitset, big), ("Solution 70x70", Solution().minimizeTheDifference, big),
                       ("set 30x30", minimize_set, mid), ("bool 30x30", minimize_bool, mid), ("bitset 30x30", minimize_bitset, mid)]:
    t0 = time.perf_counter(); r = fn(data, 800); dt = time.perf_counter() - t0
    print(f"{name:<15}: {r:>4}  {dt * 1000:7.1f} ms")

pruned 70x70   :    0     93.5 ms
bitset 70x70   :    0      1.1 ms
Solution 70x70 :    0      1.1 ms
set 30x30      :    0     28.7 ms


bool 30x30     :    0     19.2 ms
bitset 30x30   :    0      0.3 ms


## 12. Bộ test

In [16]:
ALL = {
    "set": minimize_set,
    "pruned": minimize_pruned,
    "bool": minimize_bool,
    "bitset": minimize_bitset,
    "Solution": lambda m, t: Solution().minimizeTheDifference(m, t),
}

def check(mat, target, exp=None, brute=True):
    ref = minimize_brute(mat, target) if brute else minimize_bitset(mat, target)
    for name, fn in ALL.items():
        got = fn([r[:] for r in mat], target)
        assert got == ref, (name, mat, target, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for mat_, t_, exp in examples:
    check(mat_, t_, exp)
print("Ví dụ đề: OK")

check([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 10, 2)      # tự dựng (mục 8): tổng nhỏ nhất 12
check([[70]], 1, 69)                                    # một phần tử, mọi tổng > target
check([[1] * 70 for _ in range(70)], 800, 730, brute=False)    # tổng duy nhất 70 (vét cạn 70^70 bộ: không chạy)
check([[70] * 3 for _ in range(12)], 800, 40, brute=False)     # tổng duy nhất 840
check([[5, 6], [5, 6]], 11, 0)
print("Edge case: OK")

rnd = random.Random(2001)
for _ in range(1500):
    m = rnd.randint(1, 5); n = rnd.randint(1, 4)
    mat_ = [[rnd.randint(1, 70) for _ in range(n)] for _ in range(m)]
    check(mat_, rnd.randint(1, 800))
print("1500 ma trận nhỏ so với vét cạn: OK")

for _ in range(10):
    m = rnd.randint(1, 40); n = rnd.randint(1, 40)
    mat_ = [[rnd.randint(1, 70) for _ in range(n)] for _ in range(m)]
    check(mat_, rnd.randint(1, 800), brute=False)
print("10 ma trận cỡ vừa: các cách DP khớp nhau: OK")

Ví dụ đề: OK
Edge case: OK


1500 ma trận nhỏ so với vét cạn: OK


10 ma trận cỡ vừa: các cách DP khớp nhau: OK


Các edge case trên là **tự dựng**; kỳ vọng được vét cạn xác nhận trong `check` (đối chiếu lại đề gốc nếu muốn).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam từng hàng (chọn số làm tổng gần target nhất ngay)

In [17]:
def minimize_bug_greedy(mat, target):
    s = 0
    for i, row in enumerate(mat):
        remaining_rows = len(mat) - i - 1
        # chọn x để s + x + (ước lượng tối thiểu phần còn lại) gần target
        x = min(row, key=lambda v: abs(s + v + remaining_rows - target))
        s += x
    return abs(s - target)

rnd = random.Random(3)
found = None
while found is None:
    m = rnd.randint(2, 4); n = rnd.randint(2, 4)
    mat_ = [[rnd.randint(1, 20) for _ in range(n)] for _ in range(m)]
    t_ = rnd.randint(1, 60)
    if minimize_bug_greedy(mat_, t_) != Solution().minimizeTheDifference(mat_, t_):
        found = (mat_, t_)
print("phản ví dụ:", found, "| tham lam:", minimize_bug_greedy(*found), "| đúng:", Solution().minimizeTheDifference(*found))

phản ví dụ: ([[18, 8, 7], [16, 18, 18], [16, 13, 5]], 15) | tham lam: 24 | đúng: 13


**Vì sao sai:** lựa chọn ở hàng trước giới hạn các tổng có thể đạt ở hàng sau; tham lam không nhìn trước được. **Sửa:** DP trên tập tổng.

### Lỗi 2 — Cắt tỉa bỏ **hết** các tổng > target

In [18]:
def minimize_bug_prune_all(mat, target):
    S = {0}
    for row in mat:
        S = {s + x for s in S for x in row if s + x <= target}     # SAI: có thể rỗng / bỏ ứng viên tốt
    return min((abs(s - target) for s in S), default=None)

for mat_, t_ in [([[70]], 1), ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 10)]:
    print(f"target={t_}: cắt hết = {minimize_bug_prune_all(mat_, t_)}, đúng = {Solution().minimizeTheDifference(mat_, t_)}")

target=1: cắt hết = None, đúng = 69
target=10: cắt hết = None, đúng = 2


**Vì sao sai:** đáp án tối ưu có thể là một tổng **vượt** target (ví dụ mọi tổng đều > target). **Sửa:** giữ thêm **tổng nhỏ nhất > target**.

### Lỗi 3 — Bitset quên reset `nb` cho mỗi hàng (giữ cả tổng của hàng trước)

In [19]:
def minimize_bug_bitset(mat, target):
    B = 1
    for row in mat:
        for x in set(row):
            B |= B << x                       # SAI: cho phép chọn nhiều phần tử trong một hàng, hoặc bỏ qua hàng
    sums = [s for s in range(B.bit_length()) if B >> s & 1]
    return min(abs(s - target) for s in sums)

mat_, t_ = [[1, 2, 3], [4, 5, 6], [7, 8, 9]], 3
print("sai :", minimize_bug_bitset(mat_, t_), "| đúng:", Solution().minimizeTheDifference(mat_, t_))

sai : 0 | đúng: 9


**Vì sao sai:** `B |= B << x` giữ lại cả tổng **chưa** chọn phần tử nào của hàng (bỏ qua hàng) và còn cộng dồn nhiều phần tử cùng hàng. Mỗi hàng phải chọn **đúng một** ⇒ tập mới chỉ gồm `B << x`. **Sửa:** `nb = 0; nb |= B << x; B = nb`.

### Lỗi 4 — Chỉ xét tổng ≤ target khi tính đáp án

In [20]:
def minimize_bug_only_below(mat, target):
    S = {0}
    for row in mat:
        S = {s + x for s in S for x in row}
    below = [s for s in S if s <= target]
    return target - max(below) if below else None       # SAI: bỏ qua tổng > target

mat_, t_ = [[1, 9], [1, 9]], 9
print("sai :", minimize_bug_only_below(mat_, t_), "| đúng:", Solution().minimizeTheDifference(mat_, t_))

sai : 7 | đúng: 1


**Vì sao sai:** `|·|` — tổng **lớn hơn** target cũng có thể gần hơn (`10` cách `9` một đơn vị, còn `2` cách 7). **Sửa:** lấy `min |s − target|` trên mọi `s`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn mỗi nhóm một phần tử, tối ưu hóa theo **tổng**" + giá trị nhỏ ⇒ **DP trên tập tổng** (knapsack nhóm).
- Mọi phần tử dương ⇒ cắt tỉa: tổng vượt quá ngưỡng chỉ giữ cái nhỏ nhất.
- Python: bitset bằng `int`, `B << x`, `|`, đếm/đọc bit bằng vòng lặp hoặc `bin(B)`.
- Loại trùng phần tử trong một hàng (`set(row)`) giảm số phép dịch.
- Kiểm bằng `itertools.product` trên dữ liệu nhỏ.

## 15. Biến thể của bài

1. **Tối thiểu hóa |tổng − target| khi được chọn 0 hoặc 1 phần tử mỗi hàng:** `nb = B | (B << x) ...`.
2. **LeetCode 416 — Partition Equal Subset Sum:** subset-sum bằng bitset.
3. **LeetCode 1755 — Closest Subsequence Sum:** `n ≤ 40`, giá trị lớn ⇒ meet-in-the-middle.
4. **Đếm số cách đạt mỗi tổng:** thay bitset bằng mảng đếm (convolution).
5. **Tổng không vượt target, lớn nhất có thể:** chỉ giữ tổng ≤ target, lấy max.

In [21]:
def can_partition(nums):
    """LeetCode 416 bằng bitset."""
    total = sum(nums)
    if total % 2: return False
    B = 1
    for x in nums:
        B |= B << x
    return bool(B >> (total // 2) & 1)

def count_ways_each_sum(mat):
    """Biến thể 4: số cách chọn đạt mỗi tổng."""
    ways = {0: 1}
    for row in mat:
        nw = {}
        for s, c in ways.items():
            for x in row:
                nw[s + x] = nw.get(s + x, 0) + c
        ways = nw
    return ways

print("416 [1,5,11,5] ->", can_partition([1, 5, 11, 5]), "| [1,2,3,5] ->", can_partition([1, 2, 3, 5]))
w = count_ways_each_sum([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("số cách đạt tổng 13:", w[13], "| tổng số cách:", sum(w.values()))

416 [1,5,11,5] -> True | [1,2,3,5] -> False
số cách đạt tổng 13: 3 | tổng số cách: 27


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi bộ chọn | `O(nᵐ m)` | `O(m)` | kiểm thử |
| 2. Tập tổng | `S ⊕ row` | `O(m n · mV)` | `O(mV)` | dễ hiểu |
| 3. Cắt tỉa | giữ ≤ T và một > T | `O(m n T)` | `O(T)` | Python thuần, nhanh vừa |
| 4. Mảng boolean | DP kinh điển | `O(m n · mV)` | `O(mV)` | ngôn ngữ biên dịch |
| 5. Bitset | `B << x`, `|` | `O(m n · mV/64)` | `O(mV)` bit | **nộp bài** (Python) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimizeTheDifference([[2, 4], [3, 5]], 6)` và `([[10, 20], [30, 40]], 1)`?

**Bài 2 (tính tay):** Viết tập tổng sau từng hàng cho `mat = [[1, 4], [2, 3], [5]]`, `target = 9`, và áp dụng cắt tỉa.

**Bài 3 (điền TODO):** Hoàn thiện `best_choice(mat, target)` trả về **danh sách phần tử được chọn** (mỗi hàng một) đạt chênh lệch nhỏ nhất — gợi ý: lưu "cha" của mỗi tổng.

**Bài 4 (tự viết lại):** Viết lại cách 3 (cắt tỉa) trên **mảng boolean** độ dài `target + 1` cộng một biến `above` (tổng nhỏ nhất > target).

**Bài 5 (mini challenge):** Biến thể "mỗi hàng chọn **0 hoặc 1** phần tử" — viết `minimize_optional(mat, target)` bằng bitset và kiểm bằng vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_choice(mat, target):
    # TODO: layers[i] = dict tổng -> (tổng trước, phần tử chọn); truy vết từ tổng tốt nhất
    pass

# Bài 4 — TODO
def minimize_pruned_array(mat, target):
    # TODO: f = [False]*(target+1); f[0] = True; above = inf; cập nhật từng hàng
    pass

# Bài 5 — TODO
def minimize_optional(mat, target):
    # TODO: nb = B (bỏ qua hàng) | OR (B << x)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Tổng có thể: `5, 7, 7, 9`.
- Mức 2: Gần 6 nhất là 5 hoặc 7 ⇒ 1.
- Mức 3: Ma trận thứ hai: tổng nhỏ nhất 40 ⇒ `39`.

**Bài 2**
- Mức 1: `{1, 4}` → `{3, 4, 6, 7}` → `{8, 9, 11, 12}`.
- Mức 2: Có 9 ⇒ đáp án 0.
- Mức 3: Cắt tỉa ở hàng cuối giữ `{8, 9, 11}`.

**Bài 3**
- Mức 1: Mỗi tầng lưu cho mỗi tổng một cách đạt được.
- Mức 2: Truy ngược từ tầng `m` về tầng 0.
- Mức 3: Kiểm `sum(choice)` và mỗi phần tử thuộc đúng hàng.

**Bài 4**
- Mức 1: Tổng `> target` không lưu vào mảng mà vào `above`.
- Mức 2: `above` mới = min của (`above` cũ + min hàng) và các `s + x > target`.
- Mức 3: Đáp án: min của `target − s` (s đạt) và `above − target`.

**Bài 5**
- Mức 1: "0 phần tử" = giữ nguyên tập cũ.
- Mức 2: `nb = B; for x: nb |= B << x`.
- Mức 3: Vét cạn: mỗi hàng thêm lựa chọn "không chọn" (giá trị 0).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = (Solution().minimizeTheDifference([[2, 4], [3, 5]], 6), Solution().minimizeTheDifference([[10, 20], [30, 40]], 1))
print("Bài 1:", r1)
assert r1 == (1, 39)

# Bài 2
trace_pruned([[1, 4], [2, 3], [5]], 9)

# Bài 3
def best_choice_sol(mat, target):
    layers = [{0: None}]
    for row in mat:
        cur = {}
        for s in layers[-1]:
            for x in row:
                if s + x not in cur:
                    cur[s + x] = (s, x)
        layers.append(cur)
    best = min(layers[-1], key=lambda s: abs(s - target))
    choice, s = [], best
    for i in range(len(mat), 0, -1):
        prev, x = layers[i][s]
        choice.append(x); s = prev
    return choice[::-1]

rnd = random.Random(3)
for _ in range(500):
    m = rnd.randint(1, 5); n = rnd.randint(1, 5)
    mat_ = [[rnd.randint(1, 70) for _ in range(n)] for _ in range(m)]
    t_ = rnd.randint(1, 800)
    ch = best_choice_sol(mat_, t_)
    assert all(ch[i] in mat_[i] for i in range(m)) and abs(sum(ch) - t_) == Solution().minimizeTheDifference(mat_, t_)
print("Bài 3 OK | ví dụ 1:", best_choice_sol([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 13))

Bài 1: (1, 39)
  hàng 0 [1, 4]: mọi tổng [1, 4] -> giữ [1, 4] (bỏ [])
  hàng 1 [2, 3]: mọi tổng [3, 4, 6, 7] -> giữ [3, 4, 6, 7] (bỏ [])
  hàng 2 [5]: mọi tổng [8, 9, 11, 12] -> giữ [8, 9, 11] (bỏ [12])
  đáp án: 0
Bài 3 OK | ví dụ 1: [1, 4, 8]


In [24]:
# Bài 4
def minimize_pruned_array_sol(mat, target):
    INF = float('inf')
    f = [False] * (target + 1); f[0] = True
    above = INF
    for row in mat:
        g = [False] * (target + 1)
        new_above = above + min(row) if above < INF else INF
        for s in range(target + 1):
            if f[s]:
                for x in row:
                    t = s + x
                    if t <= target:
                        g[t] = True
                    elif t < new_above:
                        new_above = t
        f, above = g, new_above
    best = above - target if above < INF else INF
    for s in range(target + 1):
        if f[s]:
            best = min(best, target - s)
    return best

# Bài 5
def minimize_optional_sol(mat, target):
    B = 1
    for row in mat:
        nb = B
        for x in set(row):
            nb |= B << x
        B = nb
    return min(abs(s - target) for s in range(B.bit_length()) if B >> s & 1)

def optional_brute(mat, target):
    return min(abs(sum(c) - target) for c in product(*[[0] + r for r in mat]))

rnd = random.Random(5)
for _ in range(800):
    m = rnd.randint(1, 4); n = rnd.randint(1, 4)
    mat_ = [[rnd.randint(1, 70) for _ in range(n)] for _ in range(m)]
    t_ = rnd.randint(1, 800)
    assert minimize_pruned_array_sol(mat_, t_) == Solution().minimizeTheDifference(mat_, t_)
    assert minimize_optional_sol(mat_, t_) == optional_brute(mat_, t_)
print("Bài 4, Bài 5 OK | optional [[10,20],[30,40]], 1 ->", minimize_optional_sol([[10, 20], [30, 40]], 1))

Bài 4, Bài 5 OK | optional [[10,20],[30,40]], 1 -> 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template knapsack nhóm bằng bitset (mỗi nhóm chọn đúng một):**

```python
B = 1                          # tổng 0
for group in groups:
    nb = 0                     # RESET cho mỗi nhóm
    for x in set(group):
        nb |= B << x
    B = nb
answer = min(abs(s - target) for s in range(B.bit_length()) if B >> s & 1)
```

**Template cắt tỉa:** giữ mọi tổng `≤ target` + **một** tổng nhỏ nhất `> target`.

**Checklist:**
- [ ] Mỗi hàng chọn **đúng một** (reset `nb`, không `B |= B << x`)?
- [ ] Xét cả tổng **lớn hơn** target khi lấy `|·|`?
- [ ] Nếu cắt tỉa: còn giữ tổng nhỏ nhất `> target`?
- [ ] Không tham lam từng hàng?
- [ ] Đã so với vét cạn `product` trên ma trận nhỏ?
- [ ] Chữ ký đúng: `def minimizeTheDifference(self, mat: List[List[int]], target: int) -> int`?